In [1]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 3.8 MB/s eta 0:00:00a 0:00:01m

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import subprocess
import shutil
import sys

# ==========================================
# 1. ENVIRONMENT DETECTION
# ==========================================
IS_COLAB = "google.colab" in sys.modules
BASE_DIR = "/content" if IS_COLAB else os.getcwd()
DOWNLOAD_DIR = os.path.join(BASE_DIR, "music_download")

print(f"Running in {'Google Colab' if IS_COLAB else 'Local Environment'}")

Running in Local Environment


In [3]:
# ==========================================
# 2. DEPENDENCY CHECK & INSTALL
# ==========================================
def setup_dependencies():
    print("Checking dependencies...")
    # Update yt-dlp
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "yt-dlp[default]"])

    if IS_COLAB:
        # Colab specific: Install Deno for signature solving
        if not shutil.which("deno"):
            print("Installing Deno (Colab)...")
            subprocess.run("curl -fsSL https://deno.land/install.sh | sh", shell=True)
            os.environ['PATH'] += ":/root/.deno/bin"
    else:
        # Local Machine: Check for essentials
        missing = [pkg for pkg in ["ffmpeg", "deno"] if not shutil.which(pkg)]
        if missing:
            print(f"WARNING: Missing local packages: {missing}")
            print("Please install them via: sudo apt install ffmpeg (for Ubuntu) or brew install (for Mac)")

setup_dependencies()

Checking dependencies...



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
# ==========================================
# 3. AUTHENTICATION (COOKIES)
# ==========================================
# Instructions for the user
print("\n--- AUTHENTICATION ---")
print("To download private playlists (like 'Your Likes'), paste your browser cookie string.")
print("If you don't need authentication (public playlists), just press Enter.")

raw_cookies = input("\nPaste 'cookie:' string (or leave blank): ").strip()
COOKIE_FILE = os.path.join(BASE_DIR, "youtube_cookies.txt")

if raw_cookies:
    if raw_cookies.startswith("cookie: "):
        raw_cookies = raw_cookies[8:]
    with open(COOKIE_FILE, "w") as f:
        f.write("# Netscape HTTP Cookie File\n")
        f.write(f".youtube.com\tTRUE\t/\tTRUE\t2147483647\tCOOKIE_DATA\t{raw_cookies}\n")
    print("Cookie file created.")
else:
    COOKIE_FILE = None
    print("Proceeding without cookies.")


--- AUTHENTICATION ---
To download private playlists (like 'Your Likes'), paste your browser cookie string.
If you don't need authentication (public playlists), just press Enter.
Proceeding without cookies.


In [11]:
import os
import subprocess

# ==========================================
# 4. INTERNAL TURBO ENGINE (INDEXING + FRAGMENTS)
# ==========================================
playlist_url = input("\nPaste the YouTube Music Playlist URL: ").strip()

ARCHIVE_FILE = os.path.join(BASE_DIR, "downloaded_history.txt")

if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)

cmd = [
    "yt-dlp", "-x", "--audio-format", "mp3", "--audio-quality", "0",
    "--js-runtime", "deno",
    "--extractor-args", "youtube:player_client=ios,web",
    "--remote-components", "ejs:github",
    "--newline",
    "--download-archive", ARCHIVE_FILE,
    "--concurrent-fragments", "5", 
    "--console-title", # <--- Forces yt-dlp to output "Downloading item X of Y"
    "-o", f"{DOWNLOAD_DIR}/%(title)s.%(ext)s",
    playlist_url
]

if COOKIE_FILE:
    cmd.extend(["--cookies", COOKIE_FILE])

print(f"\n🚀 Launching Turbo Sync... (Checking for new tracks)\n")

# Variable to keep track of the current number
current_idx = "???"

with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        # 1. Capture the index line
        if "Downloading item" in line:
            # Extracts "1 of 120"
            try:
                current_idx = line.split("item ")[1].strip()
            except:
                pass

        # 2. Check for skips
        elif "has already been recorded in the archive" in line:
            print(f"⏭️  [{current_idx}] Skipping: Already exists.")
        
        # 3. Check for downloads
        elif "[download] Destination:" in line:
            filename = os.path.basename(line.split("Destination: ")[1].strip())
            print(f"📥 [{current_idx}] Downloading: {filename}")
        
        # 4. Check for conversion
        elif "[ExtractAudio]" in line:
            print(f"🎵 [{current_idx}] Converting to MP3...")
            
        # 5. Finished track
        elif "100% of" in line and "[download]" in line:
            print(f"✅ [{current_idx}] Done.")
            print("-" * 35)

if process.returncode == 0:
    print("\n✨ Sync complete! Check the folder.")


🚀 Launching Turbo Sync... (Checking for new tracks)

⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
⏭️  [???] Skipping: Already exists.
📥 [22 of 120] Downloading: Heeriye (Official Video) Jasleen Royal ft Arijit Singh｜ Dulquer Salmaan｜ Aditya Sharma ｜Taani Tanvir.mp4
✅ [22 of 120] Done.
-----------------------------------
🎵 

KeyboardInterrupt: 

In [12]:
import concurrent.futures
import os
import subprocess
import re

# ==========================================
# 4. HIGH-SPEED PARALLEL ENGINE
# ==========================================
playlist_url = input("\nPaste the YouTube Music Playlist URL: ").strip()

# Set this between 4 and 8. Too high may cause YouTube to temporarily block your IP.
MAX_WORKERS = 6 

ARCHIVE_FILE = os.path.join(BASE_DIR, "downloaded_history.txt")
if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)

print("🔍 Rapid-scanning playlist metadata...")

# Get IDs and Titles quickly using a flat scan
scan_cmd = [
    "yt-dlp", "--flat-playlist", "--print", "%(id)s|||%(title)s", 
    playlist_url
]
if COOKIE_FILE: scan_cmd.extend(["--cookies", COOKIE_FILE])

try:
    scan_output = subprocess.check_output(scan_cmd, text=True).splitlines()
    # Remove duplicates and empty lines
    tasks = list(dict.fromkeys([line for line in scan_output if "|||" in line]))
    total = len(tasks)
    print(f"✅ Found {total} songs. Launching {MAX_WORKERS} parallel workers...\n")
except Exception as e:
    print(f"❌ Scan failed: {e}")
    tasks = []

def download_task(task_str, index):
    vid_id, title = task_str.split("|||")
    url = f"https://www.youtube.com/watch?v={vid_id}"
    
    # Minimal command for the workers
    cmd = [
        "yt-dlp", "-x", "--audio-format", "mp3", "--audio-quality", "0",
        "--js-runtime", "deno", "--extractor-args", "youtube:player_client=ios,web",
        "--download-archive", ARCHIVE_FILE, "--quiet", "--no-warnings",
        "-o", f"{DOWNLOAD_DIR}/%(title)s.%(ext)s",
        url
    ]
    if COOKIE_FILE: cmd.extend(["--cookies", COOKIE_FILE])

    # Run the download
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    # Check output to see if it was skipped or downloaded
    if "recorded in the archive" in result.stdout or "recorded in the archive" in result.stderr:
        print(f"⏭️  [{index}/{total}] Skipped: {title}")
    else:
        print(f"✅ [{index}/{total}] Finished: {title}")

# Execute parallel pool
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # We use a list comprehension to start all tasks
    future_to_song = {executor.submit(download_task, t, i+1): t for i, t in enumerate(tasks)}
    concurrent.futures.wait(future_to_song)

print("\n✨ Ultra-sync complete! Your library is up to date.")

🔍 Rapid-scanning playlist metadata...


✅ Found 120 songs. Launching 6 parallel workers...

✅ [3/120] Finished: Gehra Hua | Dhurandhar | Ranveer Singh, Sara Arjun, Shashwat Sachdev, Arijit Singh, Irshad Kamil
✅ [2/120] Finished: Sahiba (Music Video) Jasleen Royal |Vijay Deverakonda Radhikka Madan|Stebin| Priya|Aditya| Sudhanshu
✅ [4/120] Finished: Rahein Na Rahein Hum (Video) Thamma | Ayushmann K, Rashmika M | Sachin-Jigar, Soumyadeep S,Amitabh B
✅ [1/120] Finished: Uyi Amma - Azaad | Aaman D, Rasha Thadani| Madhubanti Bagchi,Amit Trivedi,Amitabh| Bosco| Abhishek K
✅ [5/120] Finished: Full Video: Raanjhan | Do Patti | Kriti Sanon, Shaheer Sheikh | Parampara Tandon | Sachet-Parampara
✅ [6/120] Finished: LAAL PARI (Song): Yo Yo Honey Singh | Sajid Nadiadwala | Tarun Mansukhani | Housefull 5
✅ [11/120] Finished: KHEL KHEL MEIN: DO U KNOW (SONG) Akshay Kumar,Diljit Dosanjh,Jaani,Ammy,Taapsee,Vaani,Tanishk
✅ [10/120] Finished: Aayi Nai -Stree 2 | Shraddha Kapoor | Rajkummar Rao | Sachin-Jigar |Pawan Singh,Simran,Divya,Amitabh
✅ [


# ==========================================
# 5. EXPORT
# ==========================================
if IS_COLAB:
    from google.colab import files
    print("\nZipping and triggering browser download...")
    shutil.make_archive('Music_Export', 'zip', DOWNLOAD_DIR)
    files.download('Music_Export.zip')
else:
    print(f"\nSuccess! Your music is located in: {DOWNLOAD_DIR}")